In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset

In [ ]:
data = pd.read_csv('diabetes.csv')
data

In [ ]:
x = data.iloc[:, 0:-1].to_numpy()
y_string = list(data.iloc[:, -1].to_numpy())

In [ ]:
y_int = []

for string in y_string:
    if string == 'positive':
        y_int.append(1)
    elif string == 'negative':
        y_int.append(0)

In [ ]:
y = np.array(y_int, dtype=np.float64)

In [ ]:
sc = StandardScaler()
x = sc.fit_transform(x)

In [ ]:
x = torch.tensor(x)
y = torch.tensor(y).unsqueeze(1)
print(x)

In [ ]:
class Dataset(Dataset):
    def __init__(self, x, y):
        self.x = x;
        self.y = y

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.x)

In [ ]:
dataset = Dataset(x, y)

In [ ]:
data_loader = torch.utils.data.dataloader.DataLoader(dataset=dataset,
                                                     batch_size=32,
                                                     shuffle=True)
for (x, y) in data_loader:
    print("Data {}".format(x.shape))
    print("Labels {}".format(y.shape))
    break

In [ ]:
class Model(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.fc1 = nn.Linear(input_features, 5)
        self.fc2 = nn.Linear(5, 4)
        self.fc3 = nn.Linear(4, 3)
        self.fc4 = nn.Linear(3, output_features)
        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()

    def forward(self, x):
        out = self.fc1(x)
        out = self.tanh(out)
        out = self.fc2(out)
        out = self.tanh(out)
        out = self.fc3(out)
        out = self.fc4(out)
        out = self.sigmoid(out)

        return out

In [ ]:
net = Model(7, 1)

criterion = nn.BCELoss(reduction='mean')
optimizer = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9)

In [ ]:
epochs = 200

for i in range(epochs):
    for (inputs, labels) in data_loader:
        inputs = inputs.float()
        labels = labels.float()

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        optimizer.step()

    output = (outputs > 0.5).float()
    accuracy = (output == labels).float().mean()
    print("Epoch: {}/{}, Loss: {:.3f}, Accuracy: {:.3f}".format(i, epochs, loss, accuracy))